In [ ]:
# Import packages
import numpy as np
import pandas as pd

# Load data
df = pd.read_parquet('data_exjobb_070425.parquet')

In [ ]:
# Split into targets and data, and remove undesired features.
dataframe = df.copy()
Targets = pd.DataFrame(dataframe['LogAdjSalePrice202006'])
to_drop = ['TransactionId', 'BaseAreaName', 'DesoArea', 'geometry', 'DistAnyCity', 'DistAnyWater', 'SalePrice', 'LogAdjSalePrice202006', 'LogSalePrice',
            'AdjSalePrice202006']
dataframe = dataframe.drop(to_drop, axis=1)

In [ ]:
# Define prepare_data function
def prepare_data(dataframe_original):
    """Takes a dataframe and finds the columns with values that are non-numerical and assigns each unique non-numerical value a numerical value."""
    dataframe = dataframe_original.copy()
    # rows = np.shape(dataframe)[0]
    cols = np.shape(dataframe)[1]
    # print(type(dataframe))
    # print(cols, rows)

    for col in range(cols):
        c = type(dataframe.iloc[0, col]) # Value of the first element in column i row 1
        # print(c)
        if c != np.float64 and c != np.int64:
            a = dataframe.iloc[:, col].unique()   # Gives the unique values of a given column
            number_of_unique_values = np.shape(a)[0]
            b = np.linspace(1, number_of_unique_values, number_of_unique_values) / number_of_unique_values
            # a.reshape(len(a),1)
            # print(np.shape(a), a, "a[0]", a[0], type(a[0]))
            
            changed_data = pd.DataFrame(dataframe[f'{dataframe.axes[1][col]}'].copy())
            # print("changed_data", changed_data, " dataframe", dataframe[f'{dataframe.axes[1][col]}'])
            # print("changed data", type(changed_data))
            for j in range(number_of_unique_values):
                # print("col",col)
                # print(dataframe[f'{dataframe.axes[1][col]}'])
                # changed_data = dataframe[f'{dataframe.axes[1][col]}'].copy()
                
                # print(j, "type", type(a[j]), type(b[j]))
                # print("value", a[j], b[j])
                if a[j] == None:
                    changed_data = changed_data.replace('None', b[j])
                else:
                    changed_data = changed_data.replace(a[j], b[j], regex=False)
                    
                    
            # print("changed data", changed_data)
            dataframe[f'{dataframe.axes[1][col]}'] = changed_data
            # print("dataframe", dataframe[f'{dataframe.axes[1][col]}'])
            
            

    return dataframe.fillna(0) # Returns the changed data with NaN values changed to zero.

In [ ]:
# Export
dataframe = dataframe.iloc[0:100, 1:15].copy()
dataframe['EstimatedContractDate'] = dataframe['EstimatedContractDate'].astype('int64') // 10**9

prepared_data = prepare_data(dataframe)
prepared_data.insert(0, "TransactionId", df['TransactionId'])

prepared_data.to_parquet('ready_data.parquet')
Targets.to_parquet('ready_targets.parquet')

/var/folders/s9/fpc42dw53pgc33tc9cwdpsbh0000gp/T/ipykernel_3095/1426284223.py:33: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  changed_data = changed_data.replace(a[j], b[j], regex=False)


In [ ]:
df2 = pd.read_parquet('ready_data.parquet')
df2.head()

,TransactionId,BuildingAge,UtilityArea,LotArea,QualityScore,CloseToBeach,EnergyPerformance,EnergyClass,HasWater,HasSewer,TaxRegionIdTp10,BaseAreaId,BuildingStyle,EstimatedContractDate
0,10006194,48,143,480,36,0,60.0,0.142857,1,1,880008,202321,2,0
1,10006232,54,120,750,32,0,114.0,0.142857,1,1,662006,210047,1,0
2,10006302,47,169,592,30,0,0.0,0.000000,1,1,586013,202312,2,0
3,10006371,128,250,3311,30,0,88.0,0.428571,1,0,680305,210044,1,0
4,10006399,112,87,3406,28,0,101.0,0.571429,1,1,883900,210046,1,0
